# LIN28A project upgrade: direct target prioritization v2

## 자율주제

**LIN28A에 강하게 결합한 mRNA가 Lin28a knockdown 후 번역 증가를 보이는지 확인하고, ER/membrane/secretory pathway 관련 후보 direct target을 우선순위화한다.**

이 노트북은 저번 과제에서 했던 `CLIP enrichment`와 `ribosome density change` 분석을 업그레이드한 버전입니다.

저번 버전과 비교해 추가된 점은 다음과 같습니다.

1. read count를 **library-size normalized CPM**으로 바꾼 뒤 비율을 계산합니다.
2. CLIP antibody replicate가 여러 개 있으면 자동으로 찾아서 **median CLIP enrichment**를 사용합니다.
3. LIN28A 결합 상위 그룹과 하위 그룹을 비교하는 것에서 끝내지 않고, **threshold sensitivity analysis**를 합니다.
4. Spearman correlation에 대해 **bootstrap confidence interval**을 계산합니다.
5. CLIP 결합이 RNA abundance 변화보다 **translation 변화와 더 관련 있는지** 비교합니다.
6. ER/membrane/secretory annotation을 붙여 **Fisher exact test**로 후보군 enrichment를 확인합니다.
7. 최종적으로 GitHub에 올릴 수 있는 `figures/`, `results/`, `summary.md`를 자동 생성합니다.

## 논문과의 연결

Cho et al. 논문은 LIN28A CLIP-seq으로 LIN28A 결합 RNA를 찾고, ribosome footprinting으로 Lin28a knockdown 후 번역 변화를 분석했습니다. 논문에서는 LIN28A가 let-7 precursor뿐 아니라 다수의 mRNA에 결합하며, 특히 ER-associated translation을 억제한다고 해석합니다. 이 노트북은 그 결론을 바탕으로 **내 데이터 분석 버전의 후보 target ranking**을 만드는 것을 목표로 합니다.

In [ ]:
# Google Colab에서 실행하는 경우 Drive를 마운트합니다.
# 로컬/서버에서 실행한다면 이 셀은 자동으로 건너뜁니다.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Colab이 아니거나 Drive가 이미 마운트되어 있습니다:', e)

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

# 이전 과제에서 사용한 기본 작업 폴더 후보들입니다.
WORKDIR_CANDIDATES = [
    Path('/content/drive/MyDrive/binfo1-work'),
    Path('/content/drive/MyDrive'),
    Path.cwd(),
]

WORKDIR = None
for p in WORKDIR_CANDIDATES:
    if p.exists() and (p / 'read-counts.txt').exists():
        WORKDIR = p
        break

if WORKDIR is None:
    # read-counts.txt가 아직 없더라도 일단 현재 폴더에서 시작합니다.
    WORKDIR = Path.cwd()

os.chdir(WORKDIR)
print('Working directory:', Path.cwd())

OUTDIR = Path('lin28a_week_upgrade_results')
FIGDIR = OUTDIR / 'figures'
RESDIR = OUTDIR / 'results'
FIGDIR.mkdir(parents=True, exist_ok=True)
RESDIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTDIR.resolve())

In [ ]:
# 필요한 Python 패키지를 확인하고 없으면 설치합니다.
required = ['pandas', 'numpy', 'matplotlib', 'scipy']
for pkg in required:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import re
import math
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu, fisher_exact

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
warnings.filterwarnings('ignore', category=RuntimeWarning)

## 1. read-counts.txt 불러오기

이전 과제에서 사용한 `read-counts.txt`를 그대로 사용합니다. `featureCounts` 결과 형태를 예상하지만, column 이름이 조금 달라도 자동으로 필요한 column을 찾도록 만들었습니다.

In [ ]:
def read_featurecounts_table(path: Path) -> pd.DataFrame:
    """featureCounts style read-count table을 유연하게 읽습니다."""
    if not path.exists():
        raise FileNotFoundError(
            f'{path} 파일이 없습니다. 이전 과제 폴더에 read-counts.txt가 있는지 확인하세요.\n'
            '보통 /content/drive/MyDrive/binfo1-work/read-counts.txt 위치에 있습니다.'
        )
    table = pd.read_csv(path, sep='\t', comment='#')
    if 'Geneid' not in table.columns:
        table = table.rename(columns={table.columns[0]: 'Geneid'})
    table['Geneid'] = table['Geneid'].astype(str)
    return table

count_file = Path('read-counts.txt')
cnts_raw = read_featurecounts_table(count_file)
print('raw shape:', cnts_raw.shape)
display(cnts_raw.head())

In [ ]:
def find_columns(columns, include_all=None, include_any=None, exclude_any=None):
    """column 이름에서 특정 문자열 조합을 포함하는 column을 찾습니다."""
    include_all = include_all or []
    include_any = include_any or []
    exclude_any = exclude_any or []
    out = []
    for c in columns:
        low = str(c).lower()
        if include_all and not all(x.lower() in low for x in include_all):
            continue
        if include_any and not any(x.lower() in low for x in include_any):
            continue
        if exclude_any and any(x.lower() in low for x in exclude_any):
            continue
        out.append(c)
    return out

def choose_one(columns, label):
    if len(columns) == 0:
        raise ValueError(f'{label}에 해당하는 column을 찾지 못했습니다.')
    if len(columns) > 1:
        print(f'[{label}] 후보가 여러 개입니다. 첫 번째를 사용합니다:', columns)
    return columns[0]

all_cols = list(cnts_raw.columns)

# count column 자동 탐색
clip_cols = find_columns(all_cols, include_all=['clip'])
rna_control_col = choose_one(find_columns(all_cols, include_all=['rna', 'control']), 'RNA-control')
rna_siluc_col = choose_one(find_columns(all_cols, include_all=['rna', 'siluc']), 'RNA-siLuc')
rna_silin28a_col = choose_one(find_columns(all_cols, include_all=['rna'], include_any=['silin28a', 'silin28', 'lin28a']), 'RNA-siLin28a')
rpf_siluc_col = choose_one(find_columns(all_cols, include_all=['rpf', 'siluc']), 'RPF-siLuc')
rpf_silin28a_col = choose_one(find_columns(all_cols, include_all=['rpf'], include_any=['silin28a', 'silin28', 'lin28a']), 'RPF-siLin28a')

if len(clip_cols) == 0:
    raise ValueError('CLIP column을 찾지 못했습니다. column 이름에 CLIP이 들어 있는지 확인하세요.')

print('CLIP columns:', clip_cols)
print('RNA-control:', rna_control_col)
print('RNA-siLuc:', rna_siluc_col)
print('RNA-siLin28a:', rna_silin28a_col)
print('RPF-siLuc:', rpf_siluc_col)
print('RPF-siLin28a:', rpf_silin28a_col)

## 2. CPM normalization 후 핵심 지표 계산

단순 raw count 비율 대신, 이번 업그레이드에서는 각 library의 total mapped count 차이를 보정하기 위해 CPM(counts per million)을 계산합니다.

계산하는 값은 다음과 같습니다.

- `clip_log2`: log2(CLIP CPM / RNA-control CPM)
- `rden_log2`: log2((RPF-siLin28a / RNA-siLin28a) / (RPF-siLuc / RNA-siLuc))
- `rna_change_log2`: log2(RNA-siLin28a / RNA-siLuc)

In [ ]:
def make_numeric(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    return df

def calculate_cpm(df, count_cols):
    counts = df[count_cols].copy()
    lib_sizes = counts.sum(axis=0)
    cpm = counts.divide(lib_sizes, axis=1) * 1_000_000
    return cpm, lib_sizes

meta_cols = [c for c in ['Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length'] if c in cnts_raw.columns]
count_cols = sorted(set(clip_cols + [rna_control_col, rna_siluc_col, rna_silin28a_col, rpf_siluc_col, rpf_silin28a_col]))

cnts = make_numeric(cnts_raw.copy(), count_cols)
cpm, lib_sizes = calculate_cpm(cnts, count_cols)
print('Library sizes')
display(lib_sizes.to_frame('total_count'))

# 분석용 dataframe
analysis = cnts[meta_cols].copy()
for c in count_cols:
    analysis[c + '_cpm'] = cpm[c].values

# low-expression filtering: RNA가 너무 적으면 비율이 불안정합니다.
min_rna_cpm = 1.0
mask = (
    (analysis[rna_control_col + '_cpm'] >= min_rna_cpm) |
    (analysis[rna_siluc_col + '_cpm'] >= min_rna_cpm) |
    (analysis[rna_silin28a_col + '_cpm'] >= min_rna_cpm)
)
analysis = analysis.loc[mask].copy()
print('genes after RNA CPM filter:', len(analysis))

pseudo_cpm = 0.1

# CLIP replicate별 enrichment 계산 후 median 사용
clip_log_cols = []
for c in clip_cols:
    new_col = f'clip_log2_{Path(str(c)).stem}'
    analysis[new_col] = np.log2((analysis[c + '_cpm'] + pseudo_cpm) / (analysis[rna_control_col + '_cpm'] + pseudo_cpm))
    clip_log_cols.append(new_col)

analysis['clip_log2'] = analysis[clip_log_cols].median(axis=1)
analysis['clip_replicate_sd'] = analysis[clip_log_cols].std(axis=1).fillna(0)

# Ribosome density = RPF / RNA
analysis['rden_siluc_log2'] = np.log2((analysis[rpf_siluc_col + '_cpm'] + pseudo_cpm) / (analysis[rna_siluc_col + '_cpm'] + pseudo_cpm))
analysis['rden_silin28a_log2'] = np.log2((analysis[rpf_silin28a_col + '_cpm'] + pseudo_cpm) / (analysis[rna_silin28a_col + '_cpm'] + pseudo_cpm))
analysis['rden_log2'] = analysis['rden_silin28a_log2'] - analysis['rden_siluc_log2']
analysis['rna_change_log2'] = np.log2((analysis[rna_silin28a_col + '_cpm'] + pseudo_cpm) / (analysis[rna_siluc_col + '_cpm'] + pseudo_cpm))

# 결측/무한대 제거
analysis = analysis.replace([np.inf, -np.inf], np.nan).dropna(subset=['clip_log2', 'rden_log2', 'rna_change_log2'])
analysis['gene_id_short'] = analysis['Geneid'].astype(str).str.split('.').str[0]

print('final genes used:', len(analysis))
display(analysis[['Geneid', 'clip_log2', 'rden_log2', 'rna_change_log2', 'clip_replicate_sd']].head())
analysis[['clip_log2', 'rden_log2', 'rna_change_log2', 'clip_replicate_sd']].describe()

## 3. CLIP enrichment와 번역 변화의 관계

논문 Figure 4D의 핵심 아이디어를 확장해서, 산점도에 binned median trend를 추가했습니다. 점 하나는 유전자 하나입니다.

In [ ]:
def savefig(name):
    path = FIGDIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches='tight')
    print('saved:', path)

# binned median trend 계산
plot_df = analysis[['clip_log2', 'rden_log2']].dropna().copy()
plot_df['bin'] = pd.qcut(plot_df['clip_log2'], q=20, duplicates='drop')
binned = plot_df.groupby('bin', observed=True).agg(
    x_mid=('clip_log2', 'median'),
    y_med=('rden_log2', 'median'),
    y_q25=('rden_log2', lambda x: np.quantile(x, 0.25)),
    y_q75=('rden_log2', lambda x: np.quantile(x, 0.75)),
    n=('rden_log2', 'size'),
).reset_index(drop=True)

rho, rho_p = spearmanr(analysis['clip_log2'], analysis['rden_log2'])

fig, ax = plt.subplots(figsize=(6.5, 5.8))
ax.scatter(analysis['clip_log2'], analysis['rden_log2'], s=8, alpha=0.22, rasterized=True)
ax.plot(binned['x_mid'], binned['y_med'], marker='o', linewidth=2, label='binned median')
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_xlabel('log2(CLIP enrichment, median of CLIP replicates)')
ax.set_ylabel('log2(ribosome density change; siLin28a / siLuc)')
ax.set_title(f'LIN28A binding vs translation change\nSpearman rho={rho:.3f}, p={rho_p:.2e}')
ax.legend(frameon=False)
savefig('fig1_clip_vs_translation_binned_scatter.png')
plt.show()

print(f'Spearman rho = {rho:.4f}, p-value = {rho_p:.3e}')

## 4. Bootstrap으로 상관계수 신뢰구간 계산

한 번의 상관계수만 제시하는 대신, 유전자를 재표본추출해서 Spearman rho의 95% confidence interval을 계산합니다.

In [ ]:
def bootstrap_spearman(x, y, n_boot=500, seed=20260526):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    y = np.asarray(y)
    n = len(x)
    rhos = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        r, _ = spearmanr(x[idx], y[idx])
        if np.isfinite(r):
            rhos.append(r)
    rhos = np.asarray(rhos)
    return {
        'rho_boot_mean': float(np.mean(rhos)),
        'rho_ci_low': float(np.quantile(rhos, 0.025)),
        'rho_ci_high': float(np.quantile(rhos, 0.975)),
        'n_boot': int(len(rhos)),
    }, rhos

boot_summary, boot_rhos = bootstrap_spearman(analysis['clip_log2'], analysis['rden_log2'])
print(boot_summary)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(boot_rhos, bins=35)
ax.axvline(rho, linewidth=2, label='observed rho')
ax.axvline(boot_summary['rho_ci_low'], linestyle='--', linewidth=1, label='95% CI')
ax.axvline(boot_summary['rho_ci_high'], linestyle='--', linewidth=1)
ax.set_xlabel('bootstrap Spearman rho')
ax.set_ylabel('frequency')
ax.set_title('Bootstrap confidence interval for correlation')
ax.legend(frameon=False)
savefig('fig2_bootstrap_spearman_ci.png')
plt.show()

## 5. RNA abundance 변화와 비교

논문에서는 LIN28A 결합이 mRNA abundance 변화보다는 ribosome occupancy 변화와 관련된다고 해석했습니다. 그래서 같은 방식으로 `clip_log2`와 `rna_change_log2`의 상관도도 계산해, translation 변화와 비교합니다.

In [ ]:
rho_rna, rho_rna_p = spearmanr(analysis['clip_log2'], analysis['rna_change_log2'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
axes[0].scatter(analysis['clip_log2'], analysis['rden_log2'], s=7, alpha=0.20, rasterized=True)
axes[0].axhline(0, linewidth=1)
axes[0].axvline(0, linewidth=1)
axes[0].set_xlabel('log2(CLIP enrichment)')
axes[0].set_ylabel('log2(ribosome density change)')
axes[0].set_title(f'Translation change\nrho={rho:.3f}')

axes[1].scatter(analysis['clip_log2'], analysis['rna_change_log2'], s=7, alpha=0.20, rasterized=True)
axes[1].axhline(0, linewidth=1)
axes[1].axvline(0, linewidth=1)
axes[1].set_xlabel('log2(CLIP enrichment)')
axes[1].set_ylabel('log2(RNA abundance change)')
axes[1].set_title(f'RNA abundance change\nrho={rho_rna:.3f}')

plt.tight_layout()
plt.savefig(FIGDIR / 'fig3_translation_vs_rna_change_control.png', dpi=220, bbox_inches='tight')
print('saved:', FIGDIR / 'fig3_translation_vs_rna_change_control.png')
plt.show()

print(f'CLIP vs ribosome density change: rho={rho:.4f}, p={rho_p:.3e}')
print(f'CLIP vs RNA abundance change:     rho={rho_rna:.4f}, p={rho_rna_p:.3e}')

## 6. Strong binder 그룹 분석: boxplot + ECDF

논문 Figure 4E처럼 LIN28A 결합 강도에 따라 유전자를 나누고, 각 그룹의 ribosome density change 분포를 비교합니다.

In [ ]:
q95 = analysis['clip_log2'].quantile(0.95)
q80 = analysis['clip_log2'].quantile(0.80)
q50 = analysis['clip_log2'].quantile(0.50)

def assign_clip_group(v):
    if v >= q95:
        return 'Top 5%'
    if v >= q80:
        return 'Top 5-20%'
    if v <= q50:
        return 'Bottom 50%'
    return 'Middle 30%'

analysis['clip_group'] = analysis['clip_log2'].apply(assign_clip_group)
group_order = ['Bottom 50%', 'Middle 30%', 'Top 5-20%', 'Top 5%']

box_data = [analysis.loc[analysis['clip_group'] == g, 'rden_log2'].values for g in group_order]
fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.boxplot(box_data, labels=group_order, showfliers=False)
ax.axhline(0, linewidth=1)
ax.set_ylabel('log2(ribosome density change)')
ax.set_title('Translation change by LIN28A binding strength')
plt.xticks(rotation=20, ha='right')
savefig('fig4_rden_by_clip_group_boxplot.png')
plt.show()

# ECDF
fig, ax = plt.subplots(figsize=(6.2, 4.8))
for g in ['Bottom 50%', 'Top 5-20%', 'Top 5%']:
    vals = np.sort(analysis.loc[analysis['clip_group'] == g, 'rden_log2'].dropna().values)
    y = np.arange(1, len(vals)+1) / len(vals)
    ax.plot(vals, y, label=f'{g} (n={len(vals)})')
ax.axvline(0, linewidth=1)
ax.set_xlabel('log2(ribosome density change)')
ax.set_ylabel('cumulative fraction')
ax.set_title('ECDF of translation change by CLIP group')
ax.legend(frameon=False)
savefig('fig5_rden_ecdf_by_clip_group.png')
plt.show()

top5 = analysis.loc[analysis['clip_group'] == 'Top 5%', 'rden_log2']
bottom50 = analysis.loc[analysis['clip_group'] == 'Bottom 50%', 'rden_log2']
stat_top_bottom, p_top_bottom = mannwhitneyu(top5, bottom50, alternative='two-sided')
median_diff_top_bottom = top5.median() - bottom50.median()
print(f'Top 5% median: {top5.median():.4f}')
print(f'Bottom 50% median: {bottom50.median():.4f}')
print(f'Median difference: {median_diff_top_bottom:.4f}')
print(f'Mann-Whitney U p-value: {p_top_bottom:.3e}')

## 7. Threshold sensitivity analysis

상위 5%라는 기준만 쓰면 임의적일 수 있으므로, top 1%, 5%, 10%, 20% 기준을 모두 비교합니다. 이 부분이 이번 업그레이드의 핵심입니다.

In [ ]:
def bh_fdr(pvals):
    """Benjamini-Hochberg FDR correction."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = ranked * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = np.minimum(q, 1.0)
    return out

thresholds = [0.01, 0.05, 0.10, 0.20]
records = []
bottom_vals = analysis.loc[analysis['clip_log2'] <= q50, 'rden_log2'].dropna()

for frac in thresholds:
    cutoff = analysis['clip_log2'].quantile(1 - frac)
    top_vals = analysis.loc[analysis['clip_log2'] >= cutoff, 'rden_log2'].dropna()
    stat, p = mannwhitneyu(top_vals, bottom_vals, alternative='two-sided')
    records.append({
        'top_fraction': frac,
        'top_percent': int(frac * 100),
        'n_top': len(top_vals),
        'top_median_rden_log2': top_vals.median(),
        'bottom50_median_rden_log2': bottom_vals.median(),
        'median_difference': top_vals.median() - bottom_vals.median(),
        'mannwhitney_p': p,
        'clip_cutoff_log2': cutoff,
    })

sensitivity = pd.DataFrame(records)
sensitivity['fdr_bh'] = bh_fdr(sensitivity['mannwhitney_p'])
sensitivity.to_csv(RESDIR / 'threshold_sensitivity.csv', index=False)
display(sensitivity)

fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.plot(sensitivity['top_percent'], sensitivity['median_difference'], marker='o', linewidth=2)
for _, row in sensitivity.iterrows():
    ax.text(row['top_percent'], row['median_difference'], f"FDR={row['fdr_bh']:.1e}", fontsize=8, ha='center', va='bottom')
ax.axhline(0, linewidth=1)
ax.set_xlabel('LIN28A CLIP enrichment top group (%)')
ax.set_ylabel('median difference vs bottom 50%')
ax.set_title('Threshold sensitivity of translation increase')
savefig('fig6_threshold_sensitivity.png')
plt.show()

## 8. Permutation test

관찰된 top 5%와 bottom 50%의 median 차이가 우연히 나올 수 있는지 확인하기 위해, CLIP label을 무작위로 섞어 permutation null distribution을 만듭니다.

In [ ]:
def permutation_test_top_vs_bottom(df, n_perm=1000, seed=20260526):
    rng = np.random.default_rng(seed)
    observed = median_diff_top_bottom
    x = df['clip_log2'].values.copy()
    y = df['rden_log2'].values.copy()
    null = []
    for _ in range(n_perm):
        perm_x = rng.permutation(x)
        hi = perm_x >= np.quantile(perm_x, 0.95)
        lo = perm_x <= np.quantile(perm_x, 0.50)
        null.append(np.nanmedian(y[hi]) - np.nanmedian(y[lo]))
    null = np.asarray(null)
    p_emp = (np.sum(np.abs(null) >= abs(observed)) + 1) / (len(null) + 1)
    return observed, null, p_emp

observed_diff, perm_null, perm_p = permutation_test_top_vs_bottom(analysis, n_perm=1000)
print(f'Observed median difference = {observed_diff:.4f}')
print(f'Permutation empirical p-value = {perm_p:.4f}')

fig, ax = plt.subplots(figsize=(6.4, 4.5))
ax.hist(perm_null, bins=40)
ax.axvline(observed_diff, linewidth=2, label='observed')
ax.axvline(0, linewidth=1)
ax.set_xlabel('median difference under permutation')
ax.set_ylabel('frequency')
ax.set_title('Permutation test: Top 5% vs Bottom 50%')
ax.legend(frameon=False)
savefig('fig7_permutation_test_top5_vs_bottom50.png')
plt.show()

## 9. Localization annotation: ER/membrane/secretory 관련성

가능하면 이전 수업자료의 `mouselocalization-20210507.txt`를 사용합니다. 파일이 없으면 URL에서 읽기를 시도합니다. 인터넷 연결이 안 되면 이 섹션은 자동으로 건너뜁니다.

In [ ]:
def load_localization_table():
    candidates = [
        Path('mouselocalization-20210507.txt'),
        Path('data/mouselocalization-20210507.txt'),
        Path('/content/drive/MyDrive/binfo1-work/mouselocalization-20210507.txt'),
    ]
    for p in candidates:
        if p.exists():
            print('localization file:', p)
            return pd.read_csv(p, sep='\t')
    url = 'https://hyeshik.qbio.io/binfo/mouselocalization-20210507.txt'
    try:
        print('localization URL에서 읽는 중:', url)
        return pd.read_csv(url, sep='\t')
    except Exception as e:
        print('localization table을 불러오지 못했습니다:', e)
        return None

local = load_localization_table()
if local is not None:
    print(local.shape)
    display(local.head())
else:
    print('localization 분석을 건너뜁니다.')

In [ ]:
def classify_er_related(text):
    text = str(text).lower()
    pattern = r'endoplasmic|\ber\b|golgi|membrane|secret|extracellular|lumen|lysosome|plasma membrane|cell surface'
    return bool(re.search(pattern, text))

merged = None
loc_p = np.nan
fisher_p = np.nan
if local is not None:
    # column 자동 추정
    gene_col_candidates = [c for c in local.columns if c.lower() in ['gene_id', 'geneid', 'ensembl_gene_id']]
    gene_col = gene_col_candidates[0] if gene_col_candidates else local.columns[0]
    type_col_candidates = [c for c in local.columns if c.lower() in ['type', 'location', 'localization', 'compartment']]
    type_col = type_col_candidates[0] if type_col_candidates else local.columns[-1]
    print('gene_col:', gene_col, 'type_col:', type_col)

    local2 = local.copy()
    local2['gene_id_short'] = local2[gene_col].astype(str).str.split('.').str[0]
    local2['location_text'] = local2[type_col].astype(str)
    local2['er_membrane_secretory'] = local2['location_text'].apply(classify_er_related)

    merged = analysis.merge(local2[['gene_id_short', 'location_text', 'er_membrane_secretory']], on='gene_id_short', how='left')
    merged['er_membrane_secretory'] = merged['er_membrane_secretory'].fillna(False)
    merged['local_group'] = np.where(merged['er_membrane_secretory'], 'ER/membrane/secretory', 'Other/unknown')
    print('merged shape:', merged.shape)
    print(merged['local_group'].value_counts())

    er_vals = merged.loc[merged['er_membrane_secretory'], 'rden_log2'].dropna()
    other_vals = merged.loc[~merged['er_membrane_secretory'], 'rden_log2'].dropna()
    if len(er_vals) > 0 and len(other_vals) > 0:
        loc_stat, loc_p = mannwhitneyu(er_vals, other_vals, alternative='two-sided')
        print(f'ER/membrane/secretory median rden_log2 = {er_vals.median():.4f}')
        print(f'Other/unknown median rden_log2 = {other_vals.median():.4f}')
        print(f'Mann-Whitney U p-value = {loc_p:.3e}')

        fig, ax = plt.subplots(figsize=(6, 4.8))
        ax.boxplot([er_vals, other_vals], labels=['ER/membrane/secretory', 'Other/unknown'], showfliers=False)
        ax.axhline(0, linewidth=1)
        ax.set_ylabel('log2(ribosome density change)')
        ax.set_title('Translation change by localization annotation')
        plt.xticks(rotation=15, ha='right')
        savefig('fig8_localization_rden_boxplot.png')
        plt.show()

        fig, ax = plt.subplots(figsize=(6.4, 5.4))
        for group, sub in merged.groupby('local_group'):
            ax.scatter(sub['clip_log2'], sub['rden_log2'], s=9, alpha=0.28, label=f'{group} (n={len(sub)})', rasterized=True)
        ax.axhline(0, linewidth=1)
        ax.axvline(0, linewidth=1)
        ax.set_xlabel('log2(CLIP enrichment)')
        ax.set_ylabel('log2(ribosome density change)')
        ax.set_title('LIN28A target pattern by localization')
        ax.legend(frameon=False)
        savefig('fig9_localization_scatter.png')
        plt.show()
else:
    merged = analysis.copy()
    merged['er_membrane_secretory'] = False
    merged['local_group'] = 'Other/unknown'

## 10. Candidate direct target scoring

후보 선정 기준은 다음과 같습니다.

1. LIN28A CLIP enrichment 상위 10%
2. Lin28a knockdown 후 ribosome density 증가: `rden_log2 > 0`
3. RNA abundance 변화가 너무 크지 않음: `abs(rna_change_log2) < 0.5`

그리고 ranking score는 결합 강도, 번역 증가, RNA 변화 penalty, ER/membrane/secretory annotation을 함께 반영합니다.

In [ ]:
def robust_zscore(s):
    s = pd.Series(s).astype(float)
    med = s.median()
    mad = np.median(np.abs(s - med))
    if mad == 0 or not np.isfinite(mad):
        std = s.std()
        if std == 0 or not np.isfinite(std):
            return pd.Series(np.zeros(len(s)), index=s.index)
        return (s - s.mean()) / std
    return 0.6745 * (s - med) / mad

rank_df = merged.copy()
rank_df['clip_z'] = robust_zscore(rank_df['clip_log2'])
rank_df['rden_z'] = robust_zscore(rank_df['rden_log2'])
rank_df['rna_abs_change'] = rank_df['rna_change_log2'].abs()
rank_df['rna_penalty_z'] = robust_zscore(rank_df['rna_abs_change']).clip(lower=0)
rank_df['location_bonus'] = np.where(rank_df.get('er_membrane_secretory', False), 0.5, 0.0)
rank_df['target_score'] = rank_df['clip_z'] + rank_df['rden_z'] - 0.5 * rank_df['rna_penalty_z'] + rank_df['location_bonus']

clip_top10_cut = rank_df['clip_log2'].quantile(0.90)
rank_df['candidate_flag'] = (
    (rank_df['clip_log2'] >= clip_top10_cut) &
    (rank_df['rden_log2'] > 0) &
    (rank_df['rna_change_log2'].abs() < 0.5)
)

candidates = rank_df.loc[rank_df['candidate_flag']].sort_values('target_score', ascending=False).copy()

candidate_cols = [
    'Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length',
    'clip_log2', 'rden_log2', 'rna_change_log2', 'clip_replicate_sd',
    'location_text', 'er_membrane_secretory',
    'clip_z', 'rden_z', 'rna_penalty_z', 'location_bonus', 'target_score'
]
candidate_cols = [c for c in candidate_cols if c in candidates.columns]

candidates[candidate_cols].to_csv(RESDIR / 'lin28a_prioritized_candidate_targets.csv', index=False)
rank_df.to_csv(RESDIR / 'lin28a_all_genes_scored.csv', index=False)
print('candidate targets:', len(candidates))
display(candidates[candidate_cols].head(30))

In [ ]:
# Candidate에서 ER/membrane/secretory gene이 background보다 풍부한지 Fisher exact test
fisher_table = None
if 'er_membrane_secretory' in rank_df.columns and rank_df['er_membrane_secretory'].nunique() > 1:
    cand_er = int(rank_df.loc[rank_df['candidate_flag'], 'er_membrane_secretory'].sum())
    cand_non = int(rank_df.loc[rank_df['candidate_flag'], 'er_membrane_secretory'].count() - cand_er)
    bg_er = int(rank_df.loc[~rank_df['candidate_flag'], 'er_membrane_secretory'].sum())
    bg_non = int(rank_df.loc[~rank_df['candidate_flag'], 'er_membrane_secretory'].count() - bg_er)
    fisher_table = np.array([[cand_er, cand_non], [bg_er, bg_non]])
    oddsratio, fisher_p = fisher_exact(fisher_table, alternative='greater')
    print('Fisher exact table [[candidate_ER, candidate_nonER], [background_ER, background_nonER]]')
    print(fisher_table)
    print(f'odds ratio = {oddsratio:.3f}, one-sided p-value = {fisher_p:.3e}')
else:
    oddsratio, fisher_p = np.nan, np.nan
    print('ER annotation이 충분하지 않아 Fisher exact test를 건너뜁니다.')

## 11. Top candidate heatmap

상위 후보 유전자들의 핵심 지표를 한눈에 보는 heatmap입니다.

In [ ]:
heat_cols = ['clip_log2', 'rden_log2', 'rna_change_log2', 'target_score']
heat = candidates.head(30).copy()
if len(heat) > 0:
    heat_mat = heat[heat_cols].copy()
    # column별 z-score로 scale하여 시각화
    heat_scaled = heat_mat.apply(lambda s: (s - s.mean()) / (s.std() if s.std() else 1), axis=0)

    fig, ax = plt.subplots(figsize=(7.2, max(4, 0.22 * len(heat))))
    im = ax.imshow(heat_scaled.values, aspect='auto')
    ax.set_xticks(range(len(heat_cols)))
    ax.set_xticklabels(heat_cols, rotation=35, ha='right')
    labels = heat['Geneid'].astype(str).str.replace(r'\.\d+$', '', regex=True).tolist()
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_title('Top candidate targets: scaled metrics')
    plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label='column-scaled value')
    savefig('fig10_top_candidate_heatmap.png')
    plt.show()
else:
    print('후보 유전자가 없어 heatmap을 만들지 않았습니다.')

## 12. 결과 요약 파일 생성

GitHub에 올렸을 때 바로 설명이 되도록, 결과 수치와 생성된 파일 목록을 `summary.md`로 저장합니다.

In [ ]:
summary = []
summary.append('# LIN28A week upgrade project summary')
summary.append('')
summary.append('## Project question')
summary.append('Do LIN28A-bound mRNAs show increased ribosome density after Lin28a knockdown, and are candidate direct targets enriched for ER/membrane/secretory genes?')
summary.append('')
summary.append('## Key results')
summary.append(f'- Number of genes analyzed after filtering: {len(analysis)}')
summary.append(f'- CLIP vs ribosome density change Spearman rho: {rho:.4f}, p={rho_p:.3e}')
summary.append(f'- Bootstrap 95% CI for rho: [{boot_summary["rho_ci_low"]:.4f}, {boot_summary["rho_ci_high"]:.4f}]')
summary.append(f'- CLIP vs RNA abundance change Spearman rho: {rho_rna:.4f}, p={rho_rna_p:.3e}')
summary.append(f'- Top 5% binder median rden_log2: {top5.median():.4f}')
summary.append(f'- Bottom 50% binder median rden_log2: {bottom50.median():.4f}')
summary.append(f'- Top 5% minus Bottom 50% median difference: {median_diff_top_bottom:.4f}')
summary.append(f'- Mann-Whitney U p-value for Top 5% vs Bottom 50%: {p_top_bottom:.3e}')
summary.append(f'- Permutation empirical p-value for median difference: {perm_p:.4f}')
summary.append(f'- Number of prioritized candidate targets: {len(candidates)}')
if np.isfinite(fisher_p):
    summary.append(f'- Fisher exact test for ER/membrane/secretory enrichment among candidates: odds ratio={oddsratio:.3f}, p={fisher_p:.3e}')
else:
    summary.append('- Fisher exact test for ER/membrane/secretory enrichment was skipped because localization annotation was unavailable or insufficient.')
summary.append('')
summary.append('## Output files')
summary.append('- results/lin28a_all_genes_scored.csv')
summary.append('- results/lin28a_prioritized_candidate_targets.csv')
summary.append('- results/threshold_sensitivity.csv')
summary.append('- figures/fig1_clip_vs_translation_binned_scatter.png')
summary.append('- figures/fig2_bootstrap_spearman_ci.png')
summary.append('- figures/fig3_translation_vs_rna_change_control.png')
summary.append('- figures/fig4_rden_by_clip_group_boxplot.png')
summary.append('- figures/fig5_rden_ecdf_by_clip_group.png')
summary.append('- figures/fig6_threshold_sensitivity.png')
summary.append('- figures/fig7_permutation_test_top5_vs_bottom50.png')
summary.append('- figures/fig8_localization_rden_boxplot.png, if localization annotation is available')
summary.append('- figures/fig9_localization_scatter.png, if localization annotation is available')
summary.append('- figures/fig10_top_candidate_heatmap.png')
summary.append('')
summary.append('## Interpretation guide')
summary.append('- Positive rden_log2 means ribosome density increased after Lin28a knockdown.')
summary.append('- If strong CLIP binders have higher rden_log2 than weak binders, this supports the idea that LIN28A normally suppresses translation of its bound mRNAs.')
summary.append('- If candidate targets are enriched for ER/membrane/secretory genes, this is consistent with the paper’s ER-associated translation model.')

summary_text = '\n'.join(summary)
(RESDIR / 'summary.md').write_text(summary_text, encoding='utf-8')
print(summary_text)

## 13. GitHub 업로드용 명령어

아래는 예시입니다. GitHub token은 노트북에 직접 적지 않는 편이 안전합니다. 업로드는 사용자가 직접 한다고 했으므로, 이 셀은 필요할 때만 수정해서 실행하면 됩니다.

In [ ]:
# GitHub에 처음 올리는 경우 예시
# USER_NAME = 'YOUR_GITHUB_USERNAME'
# REPO_NAME = 'YOUR_REPOSITORY_NAME'
#
# !git clone https://github.com/{USER_NAME}/{REPO_NAME}.git
# !cp LIN28A_week_upgrade_project.ipynb {REPO_NAME}/
# !cp -r lin28a_week_upgrade_results {REPO_NAME}/
# %cd {REPO_NAME}
# !git add .
# !git commit -m "Upgrade LIN28A direct target analysis"
# !git push origin main

# 이미 repository 폴더 안에서 실행 중이면 아래만 사용해도 됩니다.
# !git add .
# !git commit -m "Upgrade LIN28A direct target analysis"
# !git push origin main